## Vector Database Operations Test
Day 5 작업 중 Vector Database CRUD 및 검색 기능 테스트

**IMPORTANT: 사전 준비 단계**

```bash
# 1. Docker 서비스 시작 (PostgreSQL & Qdrant)
docker compose up -d

# 2. 서비스 상태 확인
docker compose ps

# 3. FastAPI 서버 실행 (별도 터미널)
uvicorn src.app.api.main:app --reload

# 4. .env 파일에 OPENAI_API_KEY 설정 확인
```

#### 테스트 대상 모듈
- `src/app/vector_db/operations.py` - VectorOperations 클래스

#### 테스트 항목
1. VectorOperations 초기화
2. 단일 아티클 삽입 (Create)
3. 배치 아티클 삽입
4. 아티클 조회 (Read)
5. 아티클 업데이트 (Update)
6. 아티클 삭제 (Delete)
7. 자연어 검색 (Semantic Search)
8. 유사 아티클 찾기 (Similar Articles)
9. 필터링 검색
10. 전체 워크플로우 시뮬레이션

In [1]:
import sys
import asyncio
import uuid
from pathlib import Path
from datetime import datetime, timezone
from dotenv import load_dotenv

# 프로젝트 루트를 Python 경로에 추가
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# 환경 변수 로드
load_dotenv(project_root / ".env")

from src.app.vector_db.operations import VectorOperations, get_vector_operations
from src.app.vector_db.client import get_qdrant_client
from src.app.processors.embedder import get_embedder
from src.app.core.config import settings

print("✓ Setup complete")
print(f"Qdrant Host: {settings.QDRANT_HOST}")
print(f"Qdrant Port: {settings.QDRANT_PORT}")
print(f"Collection Name: {settings.QDRANT_COLLECTION_NAME}")
print(f"Vector Size: {settings.QDRANT_VECTOR_SIZE}")

✓ Setup complete
Qdrant Host: localhost
Qdrant Port: 6333
Collection Name: research_articles
Vector Size: 1536


### 1. VectorOperations 초기화

VectorOperations 인스턴스를 생성하고 연결 상태를 확인합니다.

In [2]:
# VectorOperations 인스턴스 생성
ops = VectorOperations()

print("VectorOperations Configuration:")
print("=" * 60)
print(f"Collection Name: {ops.collection_name}")
print(f"Qdrant Client: {type(ops.qdrant_client).__name__}")
print(f"Embedder: {type(ops.embedder).__name__}")

# Qdrant 연결 확인
health = ops.qdrant_client.health_check()
print(f"\nQdrant Server Health:")
print(f"  Status: {health['status']}")
print(f"  Connected: {health['connected']}")

# 현재 아티클 개수
count = ops.count_articles()
print(f"\nCurrent articles count: {count}")

VectorOperations Configuration:
Collection Name: research_articles
Qdrant Client: QdrantClientWrapper
Embedder: TextEmbedder

Qdrant Server Health:
  Status: healthy
  Connected: True

Current articles count: 0


### 2. 단일 아티클 삽입 (Create)

하나의 아티클을 Vector DB에 삽입합니다.

In [3]:
print("Single Article Insertion Test:")
print("=" * 60)

# 테스트 아티클 데이터
article_id = str(uuid.uuid4())
test_article = {
    "article_id": article_id,
    "title": "Attention Is All You Need",
    "content": """
    The dominant sequence transduction models are based on complex recurrent or
    convolutional neural networks in an encoder-decoder configuration. The best
    performing models also connect the encoder and decoder through an attention
    mechanism. We propose a new simple network architecture, the Transformer,
    based solely on attention mechanisms, dispensing with recurrence and convolutions
    entirely.
    """,
    "summary": "Transformer 아키텍처를 제안하는 논문으로, 순환 신경망 없이 attention만으로 구성됩니다.",
    "source_type": "paper",
    "category": "NLP",
    "importance_score": 0.95,
    "metadata": {"authors": ["Vaswani et al."], "year": 2017},
}

Single Article Insertion Test:


In [4]:
print(f"Article ID (PostgreSQL): {article_id}")
print(f"Title: {test_article['title']}")
print(f"Category: {test_article['category']}")
print(f"Importance Score: {test_article['importance_score']}")
print("\nInserting article...")

# 아티클 삽입
vector_id = await ops.insert_article(
    article_id=test_article["article_id"],
    title=test_article["title"],
    content=test_article["content"],
    summary=test_article["summary"],
    source_type=test_article["source_type"],
    category=test_article["category"],
    importance_score=test_article["importance_score"],
    metadata=test_article["metadata"],
)

print(f"\n✅ Article inserted successfully!")
print(f"Vector ID (Qdrant): {vector_id}")
print(f"Total articles: {ops.count_articles()}")

Article ID (PostgreSQL): a9e808f3-1e09-4aa3-b151-f9e62c3c1975
Title: Attention Is All You Need
Category: NLP
Importance Score: 0.95

Inserting article...

✅ Article inserted successfully!
Vector ID (Qdrant): 7578fa35-e830-44c9-9dba-0ba0b6c19943
Total articles: 1


### 3. 배치 아티클 삽입

여러 아티클을 한 번에 삽입합니다.

In [5]:
print("Batch Article Insertion Test:")
print("=" * 60)

# 테스트 아티클들
batch_articles = [
    {
        "article_id": str(uuid.uuid4()),
        "title": "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding",
        "content": "BERT obtains new state-of-the-art results on eleven natural language processing tasks.",
        "summary": "BERT는 양방향 Transformer를 사용한 사전학습 모델입니다.",
        "source_type": "paper",
        "category": "NLP",
        "importance_score": 0.92,
        "metadata": {"authors": ["Devlin et al."], "year": 2018},
    },
    {
        "article_id": str(uuid.uuid4()),
        "title": "GPT-4 Technical Report",
        "content": "GPT-4 is a large-scale, multimodal model which can accept image and text inputs and produce text outputs.",
        "summary": "GPT-4는 텍스트와 이미지를 처리할 수 있는 대규모 멀티모달 모델입니다.",
        "source_type": "report",
        "category": "AI",
        "importance_score": 0.98,
        "metadata": {"authors": ["OpenAI"], "year": 2023},
    },
    {
        "article_id": str(uuid.uuid4()),
        "title": "Deep Residual Learning for Image Recognition",
        "content": "We present a residual learning framework to ease the training of networks that are substantially deeper than those used previously.",
        "summary": "ResNet은 잔차 연결을 통해 매우 깊은 신경망 학습을 가능하게 합니다.",
        "source_type": "paper",
        "category": "Computer Vision",
        "importance_score": 0.90,
        "metadata": {"authors": ["He et al."], "year": 2015},
    },
    {
        "article_id": str(uuid.uuid4()),
        "title": "AlphaGo: Mastering the game of Go with deep neural networks",
        "content": "We introduce a new approach to computer Go that uses deep neural networks and tree search.",
        "summary": "AlphaGo는 강화학습을 통해 바둑 게임을 마스터한 AI입니다.",
        "source_type": "paper",
        "category": "Reinforcement Learning",
        "importance_score": 0.94,
        "metadata": {"authors": ["Silver et al."], "year": 2016},
    },
    {
        "article_id": str(uuid.uuid4()),
        "title": "Generative Adversarial Networks",
        "content": "We propose a new framework for estimating generative models via an adversarial process.",
        "summary": "GAN은 생성자와 판별자가 경쟁하며 학습하는 생성 모델입니다.",
        "source_type": "paper",
        "category": "AI",
        "importance_score": 0.91,
        "metadata": {"authors": ["Goodfellow et al."], "year": 2014},
    },
]

Batch Article Insertion Test:


In [6]:
print(f"Inserting {len(batch_articles)} articles...")

# 배치 삽입
vector_ids = await ops.insert_articles_batch(batch_articles, batch_size=3)

print(f"\n✅ {len(vector_ids)} articles inserted successfully!")
print(f"\nVector IDs:")
for i, (article, vid) in enumerate(zip(batch_articles, vector_ids), 1):
    print(f"  [{i}] {article['title'][:50]}...")
    print(f"      Vector ID: {vid}")

print(f"\nTotal articles in DB: {ops.count_articles()}")

Inserting 5 articles...

✅ 5 articles inserted successfully!

Vector IDs:
  [1] BERT: Pre-training of Deep Bidirectional Transform...
      Vector ID: 0a3a51a3-ed59-4b86-acdb-9fe220094952
  [2] GPT-4 Technical Report...
      Vector ID: 6af9d7a0-d384-4338-9f40-a974248ffe83
  [3] Deep Residual Learning for Image Recognition...
      Vector ID: 23ac6ebb-34c2-47d9-bdad-8ddfee24f8f3
  [4] AlphaGo: Mastering the game of Go with deep neural...
      Vector ID: 9bc2e46a-d60e-491a-8008-d01eb07fec2c
  [5] Generative Adversarial Networks...
      Vector ID: 61970ada-f435-4708-b222-3d8f2e97a5b0

Total articles in DB: 6


### 4. 아티클 조회 (Read)

저장된 아티클을 조회합니다.

In [7]:
print("Article Retrieval Test:")
print("=" * 60)

# 단일 조회 (첫 번째로 삽입한 아티클)
print(f"[1] Retrieving single article (vector_id: {vector_id})...")
retrieved = ops.get_article(vector_id)

if retrieved:
    print(f"✅ Article retrieved:")
    print(f"  Vector ID: {retrieved['vector_id']}")
    print(f"  Article ID: {retrieved['article_id']}")
    print(f"  Title: {retrieved['title']}")
    print(f"  Summary: {retrieved['summary'][:80]}...")
    print(f"  Category: {retrieved['category']}")
    print(f"  Importance: {retrieved['importance_score']}")
    print(f"  Source Type: {retrieved['source_type']}")
else:
    print(f"❌ Article not found")

Article Retrieval Test:
[1] Retrieving single article (vector_id: 7578fa35-e830-44c9-9dba-0ba0b6c19943)...
✅ Article retrieved:
  Vector ID: 7578fa35-e830-44c9-9dba-0ba0b6c19943
  Article ID: a9e808f3-1e09-4aa3-b151-f9e62c3c1975
  Title: Attention Is All You Need
  Summary: Transformer 아키텍처를 제안하는 논문으로, 순환 신경망 없이 attention만으로 구성됩니다....
  Category: NLP
  Importance: 0.95
  Source Type: paper


In [8]:
# 배치 조회
print(f"\n[2] Retrieving multiple articles (batch)...")
batch_retrieved = ops.get_articles_batch(vector_ids[:3])

print(f"✅ Retrieved {len(batch_retrieved)} articles:")
for i, article in enumerate(batch_retrieved, 1):
    print(f"  [{i}] {article['title'][:50]}...")
    print(f"      Category: {article['category']}, Score: {article['importance_score']}")


[2] Retrieving multiple articles (batch)...
✅ Retrieved 3 articles:
  [1] BERT: Pre-training of Deep Bidirectional Transform...
      Category: NLP, Score: 0.92
  [2] GPT-4 Technical Report...
      Category: AI, Score: 0.98
  [3] Deep Residual Learning for Image Recognition...
      Category: Computer Vision, Score: 0.9


### 5. 아티클 업데이트 (Update)

아티클의 메타데이터를 업데이트합니다.

In [9]:
print("Article Update Test:")
print("=" * 60)

# [Test 1] 메타데이터만 업데이트 (임베딩 재생성 X)
print("[Test 1] Update metadata only (no embedding regeneration)...")
print(f"Vector ID: {vector_id}")

# 업데이트 전 상태
before_update = ops.get_article(vector_id)
print(f"\nBefore update:")
print(f"  Importance Score: {before_update['importance_score']}")
print(f"  Category: {before_update['category']}")

# 중요도 점수 업데이트
success = await ops.update_article(
    vector_id=vector_id,
    importance_score=0.99,
    category="Deep Learning",
    regenerate_embedding=False,  # 임베딩 재생성 안 함
)

if success:
    after_update = ops.get_article(vector_id)
    print(f"\n✅ Update successful!")
    print(f"After update:")
    print(f"  Importance Score: {after_update['importance_score']}")
    print(f"  Category: {after_update['category']}")
else:
    print(f"\n❌ Update failed")

Article Update Test:
[Test 1] Update metadata only (no embedding regeneration)...
Vector ID: 7578fa35-e830-44c9-9dba-0ba0b6c19943

Before update:
  Importance Score: 0.95
  Category: NLP

✅ Update successful!
After update:
  Importance Score: 0.99
  Category: Deep Learning


In [10]:
# [Test 2] 임베딩 재생성 (내용 변경)
print(f"\n[Test 2] Update with embedding regeneration...")
print(f"Vector ID: {vector_ids[0]}")

success = await ops.update_article(
    vector_id=vector_ids[0],
    summary="BERT는 양방향 Transformer를 사용한 혁신적인 사전학습 모델입니다. (Updated)",
    regenerate_embedding=True,  # 임베딩 재생성
)

if success:
    updated = ops.get_article(vector_ids[0])
    print(f"\n✅ Update with regeneration successful!")
    print(f"New summary: {updated['summary']}")
else:
    print(f"\n❌ Update failed")


[Test 2] Update with embedding regeneration...
Vector ID: 0a3a51a3-ed59-4b86-acdb-9fe220094952

✅ Update with regeneration successful!
New summary: BERT는 양방향 Transformer를 사용한 혁신적인 사전학습 모델입니다. (Updated)


### 6. 아티클 삭제 (Delete)

아티클을 Vector DB에서 삭제합니다.

In [11]:
print("Article Deletion Test:")
print("=" * 60)

# 삭제를 위한 테스트 아티클 추가
test_delete_id = str(uuid.uuid4())
delete_vector_id = await ops.insert_article(
    article_id=test_delete_id,
    title="Test Article for Deletion",
    content="This article will be deleted.",
    summary="삭제 테스트용 아티클입니다.",
)

Article Deletion Test:


In [12]:
print(f"Test article created: {delete_vector_id}")
print(f"Articles before deletion: {ops.count_articles()}")

# 단일 삭제
print(f"\nDeleting article...")
success = ops.delete_article(delete_vector_id)

if success:
    print(f"✅ Article deleted successfully!")
    print(f"Articles after deletion: {ops.count_articles()}")
    
    # 삭제 확인
    retrieved = ops.get_article(delete_vector_id)
    if retrieved is None:
        print(f"✅ Confirmed: Article not found after deletion")
    else:
        print(f"❌ Error: Article still exists")
else:
    print(f"❌ Deletion failed")

Article with vector_id=97ab01e4-89e0-46ea-9ed2-2afdb192f819 not found


Test article created: 97ab01e4-89e0-46ea-9ed2-2afdb192f819
Articles before deletion: 7

Deleting article...
✅ Article deleted successfully!
Articles after deletion: 6
✅ Confirmed: Article not found after deletion


In [13]:
# 배치 삭제 테스트
print(f"\n[Batch Deletion Test]")
# 삭제용 아티클 2개 생성
delete_ids = []
for i in range(2):
    vid = await ops.insert_article(
        article_id=str(uuid.uuid4()),
        title=f"Delete Test {i+1}",
        content=f"Test content {i+1}",
    )
    delete_ids.append(vid)

print(f"Created {len(delete_ids)} test articles")
print(f"Articles before batch deletion: {ops.count_articles()}")


[Batch Deletion Test]
Created 2 test articles
Articles before batch deletion: 8


In [14]:
# 배치 삭제
success = ops.delete_articles_batch(delete_ids)
if success:
    print(f"✅ Batch deletion successful!")
    print(f"Articles after batch deletion: {ops.count_articles()}")
else:
    print(f"❌ Batch deletion failed")

✅ Batch deletion successful!
Articles after batch deletion: 6


### 7. 자연어 검색 (Semantic Search)

자연어 쿼리로 유사한 아티클을 검색합니다.

In [15]:
print("Semantic Search Test:")
print("=" * 60)

# 검색 쿼리들 (영어 + 한글 혼용, 더 구체적으로)
queries = [
    "Transformer architecture for natural language processing",
    "deep learning for image recognition and computer vision",
    "reinforcement learning and game playing AI",
]

for query in queries:
    print(f"\n{'='*60}")
    print(f"Query: '{query}'")
    print(f"{'='*60}")
    
    # 검색 실행 (score_threshold 낮춤: 0.5 → 0.3)
    results = await ops.search_similar_articles(
        query=query,
        limit=3,
        score_threshold=0.3,  # 임계값 낮춤
    )
    
    print(f"\nFound {len(results)} results:")
    for i, result in enumerate(results, 1):
        print(f"\n[{i}] Score: {result['score']:.4f}")
        print(f"    Title: {result['title']}")
        print(f"    Category: {result['category']}")
        print(f"    Summary: {result['summary'][:80]}...")
        print(f"    Importance: {result['importance_score']}")

Semantic Search Test:

Query: 'Transformer architecture for natural language processing'

Found 3 results:

[1] Score: 0.4058
    Title: Attention Is All You Need
    Category: Deep Learning
    Summary: Transformer 아키텍처를 제안하는 논문으로, 순환 신경망 없이 attention만으로 구성됩니다....
    Importance: 0.99

[2] Score: 0.3715
    Title: BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding
    Category: NLP
    Summary: BERT는 양방향 Transformer를 사용한 혁신적인 사전학습 모델입니다. (Updated)...
    Importance: 0.92

[3] Score: 0.3308
    Title: GPT-4 Technical Report
    Category: AI
    Summary: GPT-4는 텍스트와 이미지를 처리할 수 있는 대규모 멀티모달 모델입니다....
    Importance: 0.98

Query: 'deep learning for image recognition and computer vision'

Found 3 results:

[1] Score: 0.5409
    Title: Deep Residual Learning for Image Recognition
    Category: Computer Vision
    Summary: ResNet은 잔차 연결을 통해 매우 깊은 신경망 학습을 가능하게 합니다....
    Importance: 0.9

[2] Score: 0.3501
    Title: AlphaGo: Mastering the game of Go with deep neu

### 8. 유사 아티클 찾기 (Similar Articles)

특정 아티클과 유사한 다른 아티클을 찾습니다.

In [16]:
print("Similar Articles Search Test:")
print("=" * 60)

# [Test 1] vector_id로 찾기
print(f"[Test 1] Find similar articles by vector_id...")
reference = ops.get_article(vector_id)
print(f"\nReference Article:")
print(f"  Vector ID: {vector_id}")
print(f"  Title: {reference['title']}")
print(f"  Category: {reference['category']}")

# 유사 아티클 검색 (score_threshold 낮춤: 0.5 → 0.3)
similar = await ops.find_similar_articles(
    vector_id=vector_id,
    limit=3,
    score_threshold=0.3,  # 임계값 낮춤
)

print(f"\nFound {len(similar)} similar articles:")
for i, article in enumerate(similar, 1):
    print(f"\n[{i}] Score: {article['score']:.4f}")
    print(f"    Title: {article['title']}")
    print(f"    Category: {article['category']}")
    print(f"    Summary: {article['summary'][:60]}...")

# [Test 2] article_id로 찾기
print(f"\n\n[Test 2] Find similar articles by article_id...")
print(f"Reference Article ID (PostgreSQL): {article_id}")

similar = await ops.find_similar_articles(
    article_id=article_id,
    limit=3,
    score_threshold=0.3,  # 임계값 낮춤
)

print(f"\nFound {len(similar)} similar articles:")
for i, article in enumerate(similar, 1):
    print(f"\n[{i}] Score: {article['score']:.4f}")
    print(f"    Title: {article['title'][:50]}...")
    print(f"    Category: {article['category']}")

Similar Articles Search Test:
[Test 1] Find similar articles by vector_id...

Reference Article:
  Vector ID: 7578fa35-e830-44c9-9dba-0ba0b6c19943
  Title: Attention Is All You Need
  Category: Deep Learning

Found 3 similar articles:

[1] Score: 0.4362
    Title: BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding
    Category: NLP
    Summary: BERT는 양방향 Transformer를 사용한 혁신적인 사전학습 모델입니다. (Updated)...

[2] Score: 0.4284
    Title: Generative Adversarial Networks
    Category: AI
    Summary: GAN은 생성자와 판별자가 경쟁하며 학습하는 생성 모델입니다....

[3] Score: 0.4216
    Title: Deep Residual Learning for Image Recognition
    Category: Computer Vision
    Summary: ResNet은 잔차 연결을 통해 매우 깊은 신경망 학습을 가능하게 합니다....


[Test 2] Find similar articles by article_id...
Reference Article ID (PostgreSQL): a9e808f3-1e09-4aa3-b151-f9e62c3c1975

Found 3 similar articles:

[1] Score: 0.4362
    Title: BERT: Pre-training of Deep Bidirectional Transform...
    Category: NLP

[2] Score: 0.4284
   

### 9. 필터링 검색

카테고리, 소스 타입, 중요도 등으로 필터링하여 검색합니다.

In [17]:
print("Filtered Search Test:")
print("=" * 60)

# [Test 1] 카테고리 필터 (score_threshold 낮춤)
print("[Test 1] Filter by category (NLP)...")
results = await ops.search_similar_articles(
    query="natural language processing and transformer models",
    limit=5,
    category=["NLP"],
    score_threshold=0.3,  # 임계값 낮춤
)

print(f"\nFound {len(results)} NLP articles:")
for i, result in enumerate(results, 1):
    print(f"  [{i}] {result['title'][:50]}... (Category: {result['category']}, Score: {result['score']:.4f})")

# [Test 2] 소스 타입 필터
print(f"\n[Test 2] Filter by source_type (paper)...")
results = await ops.search_similar_articles(
    query="deep learning research and neural networks",
    limit=5,
    source_type=["paper"],
    score_threshold=0.3,  # 임계값 낮춤
)

print(f"\nFound {len(results)} papers:")
for i, result in enumerate(results, 1):
    print(f"  [{i}] {result['title'][:50]}... (Type: {result['source_type']}, Score: {result['score']:.4f})")

# [Test 3] 중요도 점수 필터
print(f"\n[Test 3] Filter by importance score (≥ 0.9)...")
results = await ops.search_similar_articles(
    query="artificial intelligence and machine learning",
    limit=5,
    min_importance_score=0.9,
    score_threshold=0.3,  # 임계값 낮춤
)

print(f"\nFound {len(results)} high-importance articles:")
for i, result in enumerate(results, 1):
    print(f"  [{i}] {result['title'][:50]}... (Importance: {result['importance_score']}, Score: {result['score']:.4f})")

# [Test 4] 복합 필터
print(f"\n[Test 4] Multiple filters (category + source_type + importance)...")
results = await ops.search_similar_articles(
    query="neural networks and deep learning models",
    limit=5,
    category=["NLP", "AI"],
    source_type=["paper"],
    min_importance_score=0.9,
    score_threshold=0.3,  # 임계값 낮춤
)

print(f"\nFound {len(results)} filtered articles:")
for i, result in enumerate(results, 1):
    print(f"  [{i}] {result['title'][:40]}...")
    print(f"      Category: {result['category']}, Type: {result['source_type']}, Importance: {result['importance_score']}, Score: {result['score']:.4f}")

Filtered Search Test:
[Test 1] Filter by category (NLP)...

Found 1 NLP articles:
  [1] BERT: Pre-training of Deep Bidirectional Transform... (Category: NLP, Score: 0.4343)

[Test 2] Filter by source_type (paper)...

Found 5 papers:
  [1] Deep Residual Learning for Image Recognition... (Type: paper, Score: 0.5116)
  [2] AlphaGo: Mastering the game of Go with deep neural... (Type: paper, Score: 0.4505)
  [3] Attention Is All You Need... (Type: paper, Score: 0.3839)
  [4] BERT: Pre-training of Deep Bidirectional Transform... (Type: paper, Score: 0.3241)
  [5] Generative Adversarial Networks... (Type: paper, Score: 0.3177)

[Test 3] Filter by importance score (≥ 0.9)...

Found 1 high-importance articles:
  [1] AlphaGo: Mastering the game of Go with deep neural... (Importance: 0.94, Score: 0.3516)

[Test 4] Multiple filters (category + source_type + importance)...

Found 2 filtered articles:
  [1] Generative Adversarial Networks...
      Category: AI, Type: paper, Importance: 0.91, Score: 

### 10. 전체 워크플로우 시뮬레이션

실제 사용 시나리오를 시뮬레이션합니다.

In [18]:
print("Full Workflow Simulation:")
print("=" * 60)
print("""
시나리오: 새로운 논문 수집 → 저장 → 검색 → 추천

1. 새 논문을 수집하여 Vector DB에 저장
2. 사용자가 키워드로 검색
3. 검색 결과 중 하나를 선택
4. 선택한 논문과 유사한 논문 추천
5. 사용자 피드백으로 중요도 점수 업데이트
""")

# Step 1: 새 논문 저장
print("\n[Step 1] Collecting and storing new paper...")
new_paper_id = str(uuid.uuid4())
new_vector_id = await ops.insert_article(
    article_id=new_paper_id,
    title="Vision Transformer (ViT) for Image Classification",
    content="An image is worth 16x16 words: Transformers for image recognition at scale.",
    summary="Transformer를 이미지 분류에 적용한 Vision Transformer 모델입니다.",
    source_type="paper",
    category="Computer Vision",
    importance_score=0.88,
    metadata={"authors": ["Dosovitskiy et al."], "year": 2020},
)
print(f"✅ New paper stored: {new_vector_id}")

Full Workflow Simulation:

시나리오: 새로운 논문 수집 → 저장 → 검색 → 추천

1. 새 논문을 수집하여 Vector DB에 저장
2. 사용자가 키워드로 검색
3. 검색 결과 중 하나를 선택
4. 선택한 논문과 유사한 논문 추천
5. 사용자 피드백으로 중요도 점수 업데이트


[Step 1] Collecting and storing new paper...
✅ New paper stored: 902faea2-6283-4640-bf08-c0ae3d51bdf8


In [19]:
# Step 2: 사용자 검색 (score_threshold 낮춤)
print("\n[Step 2] User searches for 'Transformer models'...")
search_results = await ops.search_similar_articles(
    query="Transformer models for deep learning",
    limit=3,
    score_threshold=0.3,  # 임계값 낮춤
)

print(f"\nSearch Results ({len(search_results)} found):")
for i, result in enumerate(search_results, 1):
    print(f"\n[{i}] {result['title'][:60]}...")
    print(f"    Score: {result['score']:.4f}")
    print(f"    Category: {result['category']}")


[Step 2] User searches for 'Transformer models'...

Search Results (3 found):

[1] Attention Is All You Need...
    Score: 0.4407
    Category: Deep Learning

[2] Deep Residual Learning for Image Recognition...
    Score: 0.4288
    Category: Computer Vision

[3] Vision Transformer (ViT) for Image Classification...
    Score: 0.3896
    Category: Computer Vision


In [20]:
# Step 3: 사용자가 첫 번째 결과 선택
if search_results:
    selected = search_results[0]
    print(f"\n[Step 3] User selects: '{selected['title'][:50]}...'")
    
    # Step 4: 유사 논문 추천 (score_threshold 낮춤)
    print(f"\n[Step 4] Finding similar papers...")
    recommendations = await ops.find_similar_articles(
        vector_id=selected['vector_id'],
        limit=3,
        score_threshold=0.3,  # 임계값 낮춤
    )
    
    print(f"\nRecommendations ({len(recommendations)} papers):")
    for i, rec in enumerate(recommendations, 1):
        print(f"\n[{i}] {rec['title'][:60]}...")
        print(f"    Similarity: {rec['score']:.4f}")
        print(f"    Category: {rec['category']}")
    
    # Step 5: 사용자 피드백 반영 (중요도 점수 업데이트)
    print(f"\n[Step 5] User feedback: Increasing importance score...")
    success = await ops.update_article(
        vector_id=selected['vector_id'],
        importance_score=min(selected['importance_score'] + 0.05, 1.0),
        regenerate_embedding=False,
    )
    
    if success:
        updated = ops.get_article(selected['vector_id'])
        print(f"✅ Importance score updated:")
        print(f"   Before: {selected['importance_score']}")
        print(f"   After:  {updated['importance_score']}")
else:
    print("\n⚠️  No search results found. Skipping steps 3-5.")

print(f"\n✅ Workflow simulation complete!")
print(f"Final article count: {ops.count_articles()}")


[Step 3] User selects: 'Attention Is All You Need...'

[Step 4] Finding similar papers...

Recommendations (3 papers):

[1] Vision Transformer (ViT) for Image Classification...
    Similarity: 0.5268
    Category: Computer Vision

[2] BERT: Pre-training of Deep Bidirectional Transformers for La...
    Similarity: 0.4362
    Category: NLP

[3] Generative Adversarial Networks...
    Similarity: 0.4284
    Category: AI

[Step 5] User feedback: Increasing importance score...
✅ Importance score updated:
   Before: 0.99
   After:  1.0

✅ Workflow simulation complete!
Final article count: 7


### 11. 싱글톤 패턴 테스트

get_vector_operations() 함수의 싱글톤 패턴을 검증합니다.

In [21]:
print("Singleton Pattern Test:")
print("=" * 60)

# 여러 번 호출
ops1 = get_vector_operations()
ops2 = get_vector_operations()
ops3 = get_vector_operations()

# 같은 인스턴스인지 확인
print(f"ops1 is ops2: {ops1 is ops2}")
print(f"ops2 is ops3: {ops2 is ops3}")
print(f"ops1 is ops3: {ops1 is ops3}")

# 메모리 주소 확인
print(f"\nMemory addresses:")
print(f"  ops1: {id(ops1)}")
print(f"  ops2: {id(ops2)}")
print(f"  ops3: {id(ops3)}")

if ops1 is ops2 is ops3:
    print(f"\n✅ Singleton pattern working correctly!")
    print(f"All get_vector_operations() calls return the same instance.")
else:
    print(f"\n❌ Singleton pattern not working!")

Singleton Pattern Test:
ops1 is ops2: True
ops2 is ops3: True
ops1 is ops3: True

Memory addresses:
  ops1: 139676954765232
  ops2: 139676954765232
  ops3: 139676954765232

✅ Singleton pattern working correctly!
All get_vector_operations() calls return the same instance.


### 12. 성능 테스트

대량의 아티클 처리 성능을 측정합니다.

In [22]:
print("Performance Test:")
print("=" * 60)

import time

# 배치 삽입 성능 테스트
print("[Test 1] Batch insertion performance...")
test_articles = [
    {
        "article_id": str(uuid.uuid4()),
        "title": f"Performance Test Article {i}",
        "content": f"This is test content for performance testing. Article number {i}.",
        "summary": f"성능 테스트용 아티클 {i}번입니다.",
        "source_type": "paper",
        "category": "AI",
        "importance_score": 0.5 + (i % 5) * 0.1,
    }
    for i in range(10)
]

start = time.time()
perf_vector_ids = await ops.insert_articles_batch(test_articles, batch_size=5)
elapsed = time.time() - start

print(f"\nInserted {len(perf_vector_ids)} articles")
print(f"Time taken: {elapsed:.2f}s")
print(f"Avg time per article: {elapsed/len(perf_vector_ids):.3f}s")

# 검색 성능 테스트
print(f"\n[Test 2] Search performance...")
queries = [
    "artificial intelligence",
    "machine learning",
    "deep learning",
]

total_time = 0
for query in queries:
    start = time.time()
    results = await ops.search_similar_articles(query, limit=5)
    elapsed = time.time() - start
    total_time += elapsed
    print(f"  Query: '{query}' - {elapsed:.3f}s ({len(results)} results)")

print(f"\nAverage search time: {total_time/len(queries):.3f}s")

# 정리 (성능 테스트 데이터 삭제)
print(f"\nCleaning up performance test data...")
ops.delete_articles_batch(perf_vector_ids)
print(f"✓ Cleanup complete")

Performance Test:
[Test 1] Batch insertion performance...

Inserted 10 articles
Time taken: 1.43s
Avg time per article: 0.143s

[Test 2] Search performance...
  Query: 'artificial intelligence' - 0.355s (0 results)
  Query: 'machine learning' - 0.321s (0 results)
  Query: 'deep learning' - 0.279s (0 results)

Average search time: 0.318s

Cleaning up performance test data...
✓ Cleanup complete


### 13. 전체 테스트 요약

In [23]:
print("\n" + "=" * 80)
print("✅ Vector Database Operations 테스트 완료")
print("=" * 80)
print("""
테스트 완료된 항목:
  1. ✓ VectorOperations 초기화 및 연결 확인
  2. ✓ 단일 아티클 삽입 (insert_article)
  3. ✓ 배치 아티클 삽입 (insert_articles_batch)
  4. ✓ 아티클 조회 (get_article, get_articles_batch)
  5. ✓ 아티클 업데이트 (update_article)
     - 메타데이터만 업데이트
     - 임베딩 재생성
  6. ✓ 아티클 삭제 (delete_article, delete_articles_batch)
  7. ✓ 자연어 검색 (search_similar_articles)
  8. ✓ 유사 아티클 찾기 (find_similar_articles)
     - vector_id로 찾기
     - article_id로 찾기
  9. ✓ 필터링 검색
     - 카테고리 필터
     - 소스 타입 필터
     - 중요도 점수 필터
     - 복합 필터
 10. ✓ 전체 워크플로우 시뮬레이션
 11. ✓ 싱글톤 패턴 검증
 12. ✓ 성능 테스트

모든 테스트가 정상적으로 완료되었습니다! 🎉

주요 확인 사항:
- CRUD 작업 정상 동작
- 시맨틱 검색 및 유사도 계산 정상 작동
- 필터링 및 복합 쿼리 지원
- PostgreSQL UUID ↔ Qdrant Vector ID 매핑
- 임베딩 자동 생성 및 캐싱
""")
print("=" * 80)
print(f"\nFinal Statistics:")
print(f"  Total articles in DB: {ops.count_articles()}")
print(f"  Collection: {ops.collection_name}")


✅ Vector Database Operations 테스트 완료

테스트 완료된 항목:
  1. ✓ VectorOperations 초기화 및 연결 확인
  2. ✓ 단일 아티클 삽입 (insert_article)
  3. ✓ 배치 아티클 삽입 (insert_articles_batch)
  4. ✓ 아티클 조회 (get_article, get_articles_batch)
  5. ✓ 아티클 업데이트 (update_article)
     - 메타데이터만 업데이트
     - 임베딩 재생성
  6. ✓ 아티클 삭제 (delete_article, delete_articles_batch)
  7. ✓ 자연어 검색 (search_similar_articles)
  8. ✓ 유사 아티클 찾기 (find_similar_articles)
     - vector_id로 찾기
     - article_id로 찾기
  9. ✓ 필터링 검색
     - 카테고리 필터
     - 소스 타입 필터
     - 중요도 점수 필터
     - 복합 필터
 10. ✓ 전체 워크플로우 시뮬레이션
 11. ✓ 싱글톤 패턴 검증
 12. ✓ 성능 테스트

모든 테스트가 정상적으로 완료되었습니다! 🎉

주요 확인 사항:
- CRUD 작업 정상 동작
- 시맨틱 검색 및 유사도 계산 정상 작동
- 필터링 및 복합 쿼리 지원
- PostgreSQL UUID ↔ Qdrant Vector ID 매핑
- 임베딩 자동 생성 및 캐싱


Final Statistics:
  Total articles in DB: 7
  Collection: research_articles


### 14. 테스트 데이터 정리 (Cleanup)

⚠️ **주의**: 이 셀을 실행하면 테스트 중 생성된 모든 아티클 데이터가 삭제됩니다!

In [24]:
print("Test Data Cleanup:")
print("=" * 60)
print("⚠️  This will RECREATE the collection, deleting ALL articles from Vector DB!")
print("")

from src.app.vector_db.schema import CollectionSchema

# 현재 아티클 개수 확인
current_count = ops.count_articles()
print(f"Current article count: {current_count}")

if current_count > 0:
    print(f"\nRecreating collection to delete all data...")
    
    # 컬렉션 재생성 (모든 데이터 삭제)
    success = ops.qdrant_client.recreate_collection(
        collection_name=ops.collection_name,
        vector_size=settings.QDRANT_VECTOR_SIZE,
        distance=CollectionSchema.DISTANCE_METRIC,
    )
    
    if success:
        print(f"✅ Collection recreated successfully!")
        
        # Payload 인덱스 재생성
        print(f"\nRecreating payload indexes...")
        for index_config in CollectionSchema.PAYLOAD_INDEXES:
            ops.qdrant_client.client.create_payload_index(
                collection_name=ops.collection_name,
                field_name=index_config["field_name"],
                field_schema=index_config["field_schema"],
            )
        print(f"✅ Payload indexes recreated")
        
        # 최종 확인
        final_count = ops.count_articles()
        print(f"\nFinal article count: {final_count}")
        
        if final_count == 0:
            print(f"✅ All test data successfully deleted!")
        else:
            print(f"⚠️  Warning: {final_count} articles still remain")
    else:
        print(f"❌ Failed to recreate collection")
else:
    print(f"✓ No articles to delete")

# Qdrant 클라이언트 연결 종료
print(f"\nClosing Qdrant client connection...")
try:
    ops.qdrant_client.close()
    print(f"✅ Qdrant client connection closed")
except Exception as e:
    print(f"⚠️  Error closing client: {e}")

print("\n" + "=" * 60)
print("✓ 테스트 데이터 정리 완료")
print("\nVector DB 컬렉션이 초기화되었습니다.")
print("FastAPI 서버는 계속 실행 중입니다.")
print("=" * 60)

Test Data Cleanup:
⚠️  This will RECREATE the collection, deleting ALL articles from Vector DB!

Current article count: 7

Recreating collection to delete all data...
✅ Collection recreated successfully!

Recreating payload indexes...
✅ Payload indexes recreated

Final article count: 0
✅ All test data successfully deleted!

Closing Qdrant client connection...
✅ Qdrant client connection closed

✓ 테스트 데이터 정리 완료

Vector DB 컬렉션이 초기화되었습니다.
FastAPI 서버는 계속 실행 중입니다.


### 15. Docker Volume 완전 삭제 (선택사항)

위의 cleanup 코드는 Vector DB의 **컬렉션 데이터만** 삭제합니다. Docker 볼륨 자체를 완전히 삭제하려면 아래 명령어를 사용하세요.

#### Docker Compose 중지 및 볼륨 삭제

```bash
# 방법 1: 모든 컨테이너와 볼륨 삭제 (PostgreSQL + Qdrant 데이터 모두 삭제)
docker compose down -v

# 방법 2: Qdrant 볼륨만 선택적으로 삭제
docker compose down
docker volume rm research-curator_qdrant_data

# 방법 3: PostgreSQL은 유지하고 Qdrant만 삭제
docker compose stop qdrant
docker compose rm qdrant
docker volume rm research-curator_qdrant_data
docker compose up -d qdrant  # Qdrant 재시작
```

#### ⚠️ 주의사항

- **`docker compose down`**: 컨테이너만 중지/삭제, **볼륨은 유지** (데이터 보존)
- **`docker compose down -v`**: 컨테이너 + 볼륨 모두 삭제 (데이터 완전 삭제)
- Named volumes (`postgres_data`, `qdrant_data`)은 `docker compose down`으로는 삭제되지 않음
- PostgreSQL 데이터까지 삭제하려면 신중하게 `-v` 옵션 사용

#### 볼륨 확인 명령어

```bash
# 현재 볼륨 목록 확인
docker volume ls

# 특정 볼륨 상세 정보
docker volume inspect research-curator_qdrant_data

# 사용하지 않는 볼륨 정리
docker volume prune
```

#### 권장 워크플로우

1. **테스트 중**: Notebook의 cleanup 코드로 데이터만 삭제 (빠름)
2. **완전 초기화**: `docker compose down -v` 후 재시작 (느림, 완전 초기화)
3. **개발 완료 후**: `docker compose down`으로 컨테이너만 중지 (데이터 보존)